# MMM Recovery Bench: quickstart

Plant a known ROAS in simulated data, fit a Bayesian marketing mix model, and check whether it finds the truth.

Runs in about 3 minutes on free Colab. Change `SCENARIO` to see where recovery breaks down.

[Repo](https://github.com/vivemeasurement/mmm-recovery-bench) · by Vijay Velpula (Vive Measurement)

In [ ]:
# Colab setup: clone the repo and install PyMC-Marketing (Meridian is optional and heavier)
import os, sys
if "google.colab" in sys.modules and not os.path.exists("mmm-recovery-bench"):
    !git clone -q https://github.com/vivemeasurement/mmm-recovery-bench
    !pip install -q pymc-marketing==0.19.2
if os.path.exists("mmm-recovery-bench"):
    os.chdir("mmm-recovery-bench")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

## 1. Pick a scenario

Every scenario uses the same true adstock, saturation and effect sizes. Only the spend pattern changes:

In [ ]:
from bench.simulate import SCENARIOS, CHANNELS, CONTROLS, simulate
for s in SCENARIOS.values():
    print(f"{s.name:14s} {s.description}")

In [ ]:
SCENARIO = "correlated"   # try: clean, correlated, low_variation, short_history, endogenous
SEED = 0

df, truth = simulate(SCENARIOS[SCENARIO], SEED)
df.head()

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(CHANNELS) + 1, 1, figsize=(10, 7), sharex=True, layout="constrained")
axes[0].plot(df.date, df.sales, color="#1f1f1e"); axes[0].set_ylabel("Sales")
for ax, ch in zip(axes[1:], CHANNELS):
    ax.bar(df.date, df[ch], width=5, color="#12A4A0"); ax.set_ylabel(ch)
fig.suptitle(f"Scenario: {SCENARIO}", x=0.01, ha="left")
plt.show()

## 2. Fit PyMC-Marketing with its default priors

In [ ]:
from bench.runners import pymc_marketing
res = pymc_marketing.fit(df, CHANNELS, CONTROLS, seed=SEED, draws=500, tune=500, chains=2)
print(f"Max R-hat {res['max_rhat']:.3f} · divergences {res['divergences']}")

## 3. Score it against the truth

In [ ]:
import pandas as pd
from bench.score import score
scored = pd.DataFrame(score(res["roas_draws"], truth))
scored[["channel", "true_roas", "est_roas", "hdi_low", "hdi_high", "covered", "abs_pct_error"]].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8), layout="constrained")
for i, r in scored.iterrows():
    ax.hlines(i, r.hdi_low, r.hdi_high, color="#12A4A0", lw=6, alpha=0.35)
    ax.plot(r.est_roas, i, "o", color="#12A4A0", ms=9, label="Estimate ± 94% interval" if i == 0 else None)
    ax.plot(r.true_roas, i, "|", color="#1f1f1e", ms=22, mew=2.5, label="True ROAS" if i == 0 else None)
ax.set_yticks(range(len(scored)), scored.channel)
ax.set_title(f"ROAS recovery · {SCENARIO}", loc="left")
ax.legend(frameon=False, loc="lower right")
plt.show()

## What to take from it

- **Clean data:** the model usually lands close, and the true ROAS sits inside its interval.
- **Correlated channels or flat spend:** the data can't separate channels. Watch the intervals widen, or worse, stay narrow around the wrong answer.
- **Spend that chases demand:** no amount of sampling fixes confounding. This is where incrementality experiments earn their keep.

Full results across tools and scenarios are in the [README](https://github.com/vivemeasurement/mmm-recovery-bench#results).
Questions? Ask in [Discussions](https://github.com/vivemeasurement/mmm-recovery-bench/discussions).